In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pybedtools import BedTool

def find_overlapping_regions(bed1_df, bed2_df, bed1_cols, bed2_cols):
    bed_1 = BedTool.from_dataframe(bed1_df[bed1_cols].sort_values(bed1_cols))
    bed_2 = BedTool.from_dataframe(bed2_df[bed2_cols].sort_values(bed2_cols))
    return BedTool.to_dataframe(bed_1.intersect(bed_2, wa=True, wb=True))

def jaccard(bed1_df, bed2_df, bed1_cols, bed2_cols):
    bed_1 = BedTool.from_dataframe(bed1_df[bed1_cols].sort_values(bed1_cols))
    bed_2 = BedTool.from_dataframe(bed2_df[bed2_cols].sort_values(bed2_cols))
    return BedTool.jaccard(bed_1, bed_2)

In [ ]:
wd = os.getcwd()

In [ ]:
os.system('tar -xzvf DMRs.tar.gz')

In [ ]:
DMRs = pd.read_table("./data/403-fg-a10/DMRs_40_3_22_22_a10.bed", header=None).sort_values([1,2])
# DMRs = DMRs.loc[DMRs[6]!='nogrp']
pd.crosstab(DMRs[7],DMRs[6].apply(lambda x:x.split('|')[0]))

In [ ]:
def plotDMRlens(ab,abab,fig,ax):
    bg = '1' if ab=='403' else '2'
    DMRs = pd.read_table("./data/"+ab+"-fg-a10/DMRs_"+abab+"_a10.bed", header=None).sort_values([1,2])
    DMRs = DMRs.loc[DMRs[6]!='nogrp']
    DMRs.index = DMRs[0]+'.'+DMRs[1].astype(str)+'.'+DMRs[2].astype(str)
    DMRs[10] = DMRs[2]-DMRs[1]
    DMRs[9] = DMRs[6].apply(lambda x:'leaf' if x.find('leaf')==0 else x)
    DMRsum = DMRs.groupby([9,7]).sum()[[10]]
    
    dmrs_metilene = pd.read_table('./DMRs/metilene3.bg'+bg+'-DMRs.tsv')
    
    dmrs_wgbstools = pd.DataFrame()
    for i in pd.Series(os.listdir('./DMRs/wgbstools.bg'+bg+'-DMRs/'))\
    [pd.Series(os.listdir('./DMRs/wgbstools.bg'+bg+'-DMRs/')).apply(lambda x:((x.split('.')[0]=='Markers')&(x[0]!='.')))]:
        dmrs_wgbstools = pd.concat([dmrs_wgbstools, pd.read_table('./DMRs/wgbstools.bg'+bg+'-DMRs/'+i)])
    
    dmrs_smart = pd.read_table('./DMRs/smart2.bg'+bg+'-DMRs.txt', comment='#')
    
    dmrs_ms = pd.read_table('./DMRs/methylscore.bg'+bg+'-DMRs.bed',\
                            usecols=range(3), engine="python", comment='#', header=None).drop_duplicates()
    
    maxlen = 0
    for ii,i in enumerate([dmrs_metilene, dmrs_wgbstools, dmrs_smart, dmrs_ms, ]):
        print('#dmrs ',i.drop_duplicates(subset=list(i.columns[:3])).shape[0])
        tmp = find_overlapping_regions(DMRs,i.drop_duplicates(subset=list(i.columns[:3])),[0,1,2,3,6],list(i.columns[:3]))
        # print(tmp.shape)
        dmrstarts = sorted(tmp['thickStart'])
        tmp = tmp.groupby(['chrom','start','end','name','score']).sum(numeric_only=False).reset_index()
        tmp['simlen'] = tmp['end']-tmp['start']
        tmp['predlen'] = tmp['thickEnd']-tmp['thickStart']
    
        tmp_fn = DMRs.loc[~DMRs[1].isin(set(tmp['start']))][[0,1,2,3,6]]
        tmp_fn.columns = ['chrom','start','end','name','score']
        tmp_fn['simlen'] = tmp_fn['end'] - tmp_fn['start']
        tmp_fn['predlen'] = 0
        
        tmp_fp = i.loc[~i[i.columns[1]].isin(dmrstarts)][i.columns[:3]]
        tmp_fp.columns = ['chrom','start','end']
        tmp_fp['simlen'] = 0
        tmp_fp['predlen'] = tmp_fp['end'] - tmp_fp['start']

        print('dmrs_metilene, dmrs_wgbstools, dmrs_smart, dmrs_ms'.split(',')[ii])
        
        tmp = pd.concat([tmp,tmp_fn,tmp_fp])
        # print(tmp_fn.shape,tmp_fn.shape)
    
        import matplotlib.pyplot as plt
        # sns.lineplot(x=[-100,5100],y=[-100,5100],ax=ax[ii],linewidth=0.1,color='grey')
        
        binsize = 100

        
        print('avg. delta ',(tmp.loc[(tmp['predlen']>binsize)&(tmp['simlen']>binsize)]['simlen'] - \
             tmp.loc[(tmp['predlen']>binsize)&(tmp['simlen']>binsize)]['predlen']).apply(abs).mean())

        from sklearn.metrics import r2_score

        title_text = ''
        
        title_text += "R2-all={:.2f}".format(r2_score(tmp['simlen'],\
              tmp['predlen']))
        title_text += '\n'
        title_text += "R2-TP={:.2f}".format(r2_score(tmp.loc[(tmp['predlen']>binsize)&(tmp['simlen']>binsize)]['simlen'],\
              tmp.loc[(tmp['predlen']>binsize)&(tmp['simlen']>binsize)]['predlen']))
        title_text += '\n'
        title_text += str(('green',len(tmp.loc[(tmp['predlen']>binsize)&(tmp['simlen']>binsize)].index)))
        title_text += '\n'
        title_text += str(('red',len(tmp.loc[(tmp['predlen']<binsize)&(tmp['simlen']>binsize)].index)))
        title_text += '\n'
        title_text += str(('blue',len(tmp.loc[(tmp['predlen']>binsize)&(tmp['simlen']<binsize)].index)))
        print()
        sns.histplot(x=tmp.loc[(tmp['predlen']>binsize)&(tmp['simlen']>binsize)]['simlen'],\
                     y=tmp.loc[(tmp['predlen']>binsize)&(tmp['simlen']>binsize)]['predlen'],\
                     color='green', binwidth=binsize, cbar=False, vmin=1, vmax=100, ax=ax[ii])
        sns.histplot(x=tmp.loc[(tmp['predlen']>binsize)&(tmp['simlen']<binsize)]['simlen'],\
                     y=tmp.loc[(tmp['predlen']>binsize)&(tmp['simlen']<binsize)]['predlen'],\
                     color='blue', binwidth=binsize, cbar=False, vmin=1, vmax=100, ax=ax[ii])
        sns.histplot(x=tmp.loc[(tmp['predlen']<binsize)&(tmp['simlen']>binsize)]['simlen'],\
                     y=tmp.loc[(tmp['predlen']<binsize)&(tmp['simlen']>binsize)]['predlen'],\
                     color='red', binwidth=binsize, cbar=False, vmin=1, vmax=100, ax=ax[ii])

        ax[ii].set_title(title_text)
        maxlen = max(maxlen, tmp['simlen'].max())
        maxlen = max(maxlen, tmp['predlen'].max())

        ax[ii].set_aspect('equal')

    sns.histplot(x=tmp['simlen'],\
                 y=tmp['predlen'],\
                 color='green', binwidth=binsize, cbar=True, vmin=1, vmax=100, ax=ax[ii+1])
    sns.histplot(x=tmp['simlen'],\
                 y=tmp['predlen'],\
                 color='blue', binwidth=binsize, cbar=True, vmin=1, vmax=100, ax=ax[ii+1])
    sns.histplot(x=tmp['simlen'],\
                 y=tmp['predlen'],\
                 color='red', binwidth=binsize, cbar=True, vmin=1, vmax=100, ax=ax[ii+1])
    ax[ii+1].set_aspect('equal')

    print('maxlen',maxlen)
    # plt.xlim([-100,(int(maxlen/1000)+1)*1000])
    # plt.ylim([-100,(int(maxlen/1000)+1)*1000])
    plt.xlim([-100,5100])
    plt.ylim([-100,5100])
    plt.xticks([0,2500,5000])
    plt.yticks([0,2500,5000])
    # return tmp

In [ ]:
ab='403'
abab = '40_3_22_22'

import matplotlib.pyplot as plt
fig, ax = plt.subplots(2,5,figsize=[12,12],sharey=True, sharex=True)
plotDMRlens(ab,abab,fig,ax[0])

ab='155'
abab = '15_5_10_10'
plotDMRlens(ab,abab,fig,ax[1])
plt.savefig('./figures/2d.pdf')

In [ ]:
def plotDMRsens(ab,abab,ax,ix):
    bg = '1' if ab=='403' else '2'
    DMRs = pd.read_table("./data/"+ab+"-fg-a10/DMRs_"+abab+"_a10.bed", header=None).sort_values([1,2])
    DMRs = DMRs.loc[DMRs[6]!='nogrp']
    DMRs.index = DMRs[0]+'.'+DMRs[1].astype(str)+'.'+DMRs[2].astype(str)
    DMRs[10] = DMRs[2]-DMRs[1]
    DMRs[9] = DMRs[6].apply(lambda x:'leaf' if x.find('leaf')==0 else x)
    DMRsum = DMRs.groupby([9,7]).sum()[[10]]
    
    dmrs_metilene = pd.read_table('./DMRs/metilene3.bg'+bg+'-DMRs.tsv')
    
    dmrs_wgbstools = pd.DataFrame()
    for i in pd.Series(os.listdir('./DMRs/wgbstools.bg'+bg+'-DMRs/'))\
    [pd.Series(os.listdir('./DMRs/wgbstools.bg'+bg+'-DMRs/')).apply(lambda x:((x.split('.')[0]=='Markers')&(x[0]!='.')))]:
        dmrs_wgbstools = pd.concat([dmrs_wgbstools, pd.read_table('./DMRs/wgbstools.bg'+bg+'-DMRs/'+i)])
    
    dmrs_smart = pd.read_table('./DMRs/smart2.bg'+bg+'-DMRs.txt', comment='#')
    
    dmrs_ms = pd.read_table('./DMRs/methylscore.bg'+bg+'-DMRs.bed',\
                            usecols=range(3), engine="python", comment='#', header=None).drop_duplicates()


    scores = {'metilene': {},'smart': {},'wgbstools': {},'ms': {}}
    for i in DMRsum.index:
        dt = i[0]
        dc = i[1]
        scores['metilene'][(dt,dc)] = jaccard(DMRs.loc[(DMRs[9]==dt)&(DMRs[7]==dc)],dmrs_metilene,[0,1,2],'chr	start	stop'.split('\t'))['intersection']
        scores['smart'][(dt,dc)] = jaccard(DMRs.loc[(DMRs[9]==dt)&(DMRs[7]==dc)],dmrs_smart,[0,1,2],'Chrome	Start	End'.split('\t'))['intersection']
        scores['wgbstools'][(dt,dc)] = jaccard(DMRs.loc[(DMRs[9]==dt)&(DMRs[7]==dc)],dmrs_wgbstools,[0,1,2],'#chr	start	end'.split('\t'))['intersection']
        scores['ms'][(dt,dc)] = jaccard(DMRs.loc[(DMRs[9]==dt)&(DMRs[7]==dc)],dmrs_ms,[0,1,2],[0,1,2])['intersection']
    
    for i in ['metilene','smart','wgbstools',]:
        DMRsum[i] = DMRsum.index.map(scores[i])/DMRsum[10]
    
    DMRsum['type'] = [i[0] for i in DMRsum.index]
    DMRsum['c'] = [1-i[1] for i in DMRsum.index]
    DMRsum['bg'] = ab
    
    import seaborn as sns
    
    color = {'metilene':'#009444', 
             'ms':'#676598',
             'wgbstools':'#939598',
             'smart':'#a97c50',}
    for ii,i in enumerate(['0|others','0,1,2|3,4','0,1|2','3|4','leaf']):    
        
        for j in ['metilene','smart','wgbstools',]:
            for k in DMRsum['bg'].unique():
                sns.lineplot(data=DMRsum.loc[(DMRsum['type']==i)&(DMRsum['bg']==k)],\
                             x='c',y=j,color=color[j], ax=ax[ii][ix])
        
        ax[ii][ix].set_ylim([-0.1,1.1])
        ax[ii][ix].set_yticks([0,0.5,1])
        ax[ii][ix].set_xticks([1-1,1-0.87,1-0.73,1-0.6])
        # ax[ii][ix].set_title(i)
        ax[ii][ix].set_ylabel(None)
        ax[ii][ix].set_xlabel(None)
        ax[ii][ix].spines['top'].set_visible(False)
        
        ax[ii][ix].spines['right'].set_visible(False)
        if i!='leaf':
            ax[ii][ix].set_xticks([])
            ax[ii][ix].spines['bottom'].set_visible(False)

In [ ]:
ab='403'
abab = '40_3_22_22'

import matplotlib.pyplot as plt
fig, ax = plt.subplots(5,2,figsize=[3,7],)

plotDMRsens(ab,abab,ax,0)

ab='155'
abab = '15_5_10_10'
plotDMRsens(ab,abab,ax,1)
plt.savefig('./figures/2b.pdf')

In [ ]:
def plotDMRsens(ab,abab,ax,ix):
    bg = '1' if ab=='403' else '2'
    DMRs = pd.read_table("./data/"+ab+"-fg-a10/DMRs_"+abab+"_a10.bed", header=None).sort_values([1,2])
    DMRs = DMRs.loc[DMRs[6]!='nogrp']
    DMRs.index = DMRs[0]+'.'+DMRs[1].astype(str)+'.'+DMRs[2].astype(str)
    DMRs[10] = DMRs[2]-DMRs[1]
    DMRs[9] = DMRs[6].apply(lambda x:'leaf' if x.find('leaf')==0 else x)
    DMRsum = DMRs.groupby([9,7]).sum()[[10]]
    
    dmrs_metilene = pd.read_table('./DMRs/metilene3.bg'+bg+'-DMRs.tsv')
    
    dmrs_wgbstools = pd.DataFrame()
    for i in pd.Series(os.listdir('./DMRs/wgbstools.bg'+bg+'-DMRs/'))\
    [pd.Series(os.listdir('./DMRs/wgbstools.bg'+bg+'-DMRs/')).apply(lambda x:((x.split('.')[0]=='Markers')&(x[0]!='.')))]:
        dmrs_wgbstools = pd.concat([dmrs_wgbstools, pd.read_table('./DMRs/wgbstools.bg'+bg+'-DMRs/'+i)])
    
    dmrs_smart = pd.read_table('./DMRs/smart2.bg'+bg+'-DMRs.txt', comment='#')
    
    dmrs_ms = pd.read_table('./DMRs/methylscore.bg'+bg+'-DMRs.bed',\
                            usecols=range(3), engine="python", comment='#', header=None).drop_duplicates()



    scores = {'metilene': {},'smart': {},'wgbstools': {},'ms': {}}
    for i in DMRsum.index:
        dt = i[0]
        dc = i[1]
        scores['metilene'][(dt,dc)] = jaccard(DMRs.loc[(DMRs[9]==dt)&(DMRs[7]==dc)],dmrs_metilene,[0,1,2],'chr	start	stop'.split('\t'))['intersection']
        scores['smart'][(dt,dc)] = jaccard(DMRs.loc[(DMRs[9]==dt)&(DMRs[7]==dc)],dmrs_smart,[0,1,2],'Chrome	Start	End'.split('\t'))['intersection']
        scores['wgbstools'][(dt,dc)] = jaccard(DMRs.loc[(DMRs[9]==dt)&(DMRs[7]==dc)],dmrs_wgbstools,[0,1,2],'#chr	start	end'.split('\t'))['intersection']
        scores['ms'][(dt,dc)] = jaccard(DMRs.loc[(DMRs[9]==dt)&(DMRs[7]==dc)],dmrs_ms,[0,1,2],[0,1,2])['intersection']
    
    for i in ['metilene','ms',]:
        DMRsum[i] = DMRsum.index.map(scores[i])/DMRsum[10]
    
    DMRsum['type'] = [i[0] for i in DMRsum.index]
    DMRsum['c'] = [1-i[1] for i in DMRsum.index]
    DMRsum['bg'] = ab
    
    import seaborn as sns
    
    color = {'metilene':'#009444', 
             'ms':'#676598',
             'wgbstools':'#939598',
             'smart':'#a97c50',}
    for ii,i in enumerate(['0|others','0,1,2|3,4','0,1|2','3|4','leaf']):    
        
        for j in ['metilene','ms',]:
            for k in DMRsum['bg'].unique():
                sns.lineplot(data=DMRsum.loc[(DMRsum['type']==i)&(DMRsum['bg']==k)],\
                             x='c',y=j,color=color[j], ax=ax[ii][ix])
        
        ax[ii][ix].set_ylim([-0.1,1.1])
        ax[ii][ix].set_yticks([0,0.5,1])
        ax[ii][ix].set_xticks([1-1,1-0.87,1-0.73,1-0.6])
        # ax[ii][ix].set_title(i)
        ax[ii][ix].set_ylabel(None)
        ax[ii][ix].set_xlabel(None)
        ax[ii][ix].spines['top'].set_visible(False)
        
        ax[ii][ix].spines['right'].set_visible(False)
        if i!='leaf':
            ax[ii][ix].set_xticks([])
            ax[ii][ix].spines['bottom'].set_visible(False)

In [ ]:
ab='403'
abab = '40_3_22_22'

import matplotlib.pyplot as plt
fig, ax = plt.subplots(5,2,figsize=[3,7],)

plotDMRsens(ab,abab,ax,0)

ab='155'
abab = '15_5_10_10'
plotDMRsens(ab,abab,ax,1)
plt.savefig('./figures/ED2h.pdf')

In [ ]:
ab='403'
abab = '40_3_22_22'

bg = '1' if ab=='403' else '2'
DMRs = pd.read_table("./data/"+ab+"-fg-a10/DMRs_"+abab+"_a10.bed", header=None).sort_values([1,2])
DMRs = DMRs.loc[DMRs[6]!='nogrp']
DMRs.index = DMRs[0]+'.'+DMRs[1].astype(str)+'.'+DMRs[2].astype(str)
DMRs[10] = DMRs[2]-DMRs[1]
DMRs[9] = DMRs[6].apply(lambda x:'leaf' if x.find('leaf')==0 else x)
DMRsum = DMRs.groupby([9,7]).sum()[[10]]

dmrs_ms = pd.read_table('./DMRs/methylscore.bg'+bg+'-DMRs.bed',\
                            usecols=range(3), engine="python", comment='#', header=None).drop_duplicates()

tmp_overlap = find_overlapping_regions(DMRs,dmrs_ms,[0,1,2,6,7],[0,1,2])
100*(1-(tmp_overlap['score'].value_counts().sort_index()/DMRs[7].value_counts().sort_index()))

In [ ]:
ab='155'
abab = '15_5_10_10'

bg = '1' if ab=='403' else '2'
DMRs = pd.read_table("./data/"+ab+"-fg-a10/DMRs_"+abab+"_a10.bed", header=None).sort_values([1,2])
DMRs = DMRs.loc[DMRs[6]!='nogrp']
DMRs.index = DMRs[0]+'.'+DMRs[1].astype(str)+'.'+DMRs[2].astype(str)
DMRs[10] = DMRs[2]-DMRs[1]
DMRs[9] = DMRs[6].apply(lambda x:'leaf' if x.find('leaf')==0 else x)
DMRsum = DMRs.groupby([9,7]).sum()[[10]]

dmrs_ms = pd.read_table('./DMRs/methylscore.bg'+bg+'-DMRs.bed',\
                            usecols=range(3), engine="python", comment='#', header=None).drop_duplicates()

tmp_overlap = find_overlapping_regions(DMRs,dmrs_ms,[0,1,2,6,7],[0,1,2])
100*(1-(tmp_overlap['score'].value_counts().sort_index()/DMRs[7].value_counts().sort_index()))

In [ ]:
ab='403'
abab = '40_3_22_22'

bg = '1' if ab=='403' else '2'
DMRs = pd.read_table("./data/"+ab+"-fg-a10/DMRs_"+abab+"_a10.bed", header=None).sort_values([1,2])
DMRs = DMRs.loc[DMRs[6]!='nogrp']
DMRs.index = DMRs[0]+'.'+DMRs[1].astype(str)+'.'+DMRs[2].astype(str)
DMRs[10] = DMRs[2]-DMRs[1]
DMRs[9] = DMRs[6].apply(lambda x:'leaf' if x.find('leaf')==0 else x)
DMRsum = DMRs.groupby([9,7]).sum()[[10]]

dmrs_metilene = pd.read_table('./DMRs/metilene3.bg'+bg+'-DMRs.tsv')

tmp_overlap = jaccard(DMRs,dmrs_metilene,[0,1,2],'chr	start	stop'.split('\t'))
tmp_overlap['intersection']/DMRs[10].sum(),\
tmp_overlap['intersection']/(dmrs_metilene['stop']-dmrs_metilene['start']).sum()

In [ ]:
ab='155'
abab = '15_5_10_10'

bg = '1' if ab=='403' else '2'
DMRs = pd.read_table("./data/"+ab+"-fg-a10/DMRs_"+abab+"_a10.bed", header=None).sort_values([1,2])
DMRs = DMRs.loc[DMRs[6]!='nogrp']
DMRs.index = DMRs[0]+'.'+DMRs[1].astype(str)+'.'+DMRs[2].astype(str)
DMRs[10] = DMRs[2]-DMRs[1]
DMRs[9] = DMRs[6].apply(lambda x:'leaf' if x.find('leaf')==0 else x)
DMRsum = DMRs.groupby([9,7]).sum()[[10]]

dmrs_metilene = pd.read_table('./DMRs/metilene3.bg'+bg+'-DMRs.tsv')

tmp_overlap = jaccard(DMRs,dmrs_metilene,[0,1,2],'chr	start	stop'.split('\t'))
tmp_overlap['intersection']/DMRs[10].sum(),\
tmp_overlap['intersection']/(dmrs_metilene['stop']-dmrs_metilene['start']).sum()

In [ ]:
def getjaccard(ab,abab):
    bg = '1' if ab=='403' else '2'
    DMRs = pd.read_table("./data/"+ab+"-fg-a10/DMRs_"+abab+"_a10.bed", header=None).sort_values([1,2])
    DMRs = DMRs.loc[DMRs[6]!='nogrp']
    DMRs.index = DMRs[0]+'.'+DMRs[1].astype(str)+'.'+DMRs[2].astype(str)
    DMRs[10] = DMRs[2]-DMRs[1]
    DMRs[9] = DMRs[6].apply(lambda x:'leaf' if x.find('leaf')==0 else x)
    DMRsum = DMRs.groupby([9,7]).sum()[[10]]
    
    dmrs_metilene = pd.read_table('./DMRs/metilene3.bg'+bg+'-DMRs.tsv')
    
    dmrs_wgbstools = pd.DataFrame()
    for i in pd.Series(os.listdir('./DMRs/wgbstools.bg'+bg+'-DMRs/'))\
    [pd.Series(os.listdir('./DMRs/wgbstools.bg'+bg+'-DMRs/')).apply(lambda x:((x.split('.')[0]=='Markers')&(x[0]!='.')))]:
        dmrs_wgbstools = pd.concat([dmrs_wgbstools, pd.read_table('./DMRs/wgbstools.bg'+bg+'-DMRs/'+i)])
    
    dmrs_smart = pd.read_table('./DMRs/smart2.bg'+bg+'-DMRs.txt', comment='#')
    
    dmrs_ms = pd.read_table('./DMRs/methylscore.bg'+bg+'-DMRs.bed',\
                            usecols=range(3), engine="python", comment='#', header=None).drop_duplicates()

    jaccard_dict = {}
    jaccard_dict['metilene'] = jaccard(DMRs,dmrs_metilene,[0,1,2],'chr	start	stop'.split('\t'))['jaccard']
    jaccard_dict['wgbstools'] = jaccard(DMRs,dmrs_wgbstools,[0,1,2],'#chr	start	end'.split('\t'))['jaccard']
    jaccard_dict['smart'] = jaccard(DMRs,dmrs_smart,[0,1,2],'Chrome	Start	End'.split('\t'))['jaccard']
    # jaccard_dict['ms'] = jaccard(DMRs,dmrs_ms,[0,1,2],[0,1,2])['jaccard']
    return jaccard_dict

In [ ]:
ab='403'
abab = '40_3_22_22'
jaccard_dict = getjaccard(ab,abab)

ab='155'
abab = '15_5_10_10'
jaccardlow_dict = getjaccard(ab,abab)

In [ ]:
jaccard_all = pd.DataFrame(pd.Series(jaccard_dict))
jaccard_all[1] = '403'
jaccard_all = pd.concat([pd.DataFrame(pd.Series(jaccardlow_dict)), jaccard_all])
jaccard_all[1] = jaccard_all[1].fillna('155')
jaccard_all

In [ ]:
fig, ax = plt.subplots()
sns.barplot(x=jaccard_all.index,y=jaccard_all[0],hue=jaccard_all[1],\
            palette={'403':sns.color_palette("Paired")[1],'155':sns.color_palette("Paired")[0]},hue_order=['403','155'])
plt.yticks([0,0.5,1])
plt.savefig('./figures/2c.pdf')

In [ ]:
ab='403'
abab = '40_3_22_22'
bg = '1' if ab=='403' else '2'

In [ ]:
DMRs = pd.read_table("./data/"+ab+"-fg-a10/DMRs_"+abab+"_a10.bed", header=None).sort_values([1,2])
DMRs = DMRs.loc[DMRs[6]!='nogrp']
DMRs.index = DMRs[0]+'.'+DMRs[1].astype(str)+'.'+DMRs[2].astype(str)
DMRs['type'] = DMRs[6].apply(lambda x:('0,1,2|3,4L' if x[:4]=='leaf' else x))
DMRs[6] = DMRs[6]+(DMRs[3]>0).map({True:'(P)',False:'(N)'}) +'@'+(DMRs[7]).astype(str)
DMRs['leaf'] = DMRs[6].apply(lambda x:(x.split('|')[1] if x[:4]=='leaf' else None))
DMRs['c'] = DMRs[7]
DMRs['len'] = DMRs[2]-DMRs[1]
DMRs['mdiff'] = DMRs[3]
Predictions = pd.read_table('./DMRs/metilene3.bg'+bg+'-DMRs.tsv')

from pybedtools import BedTool
def find_overlapping_regions_df(bed1_df, bed2_df, bed1_cols, bed2_cols):
    bed_1 = BedTool.from_dataframe(bed1_df[bed1_cols].sort_values(bed1_cols))
    bed_2 = BedTool.from_dataframe(bed2_df[bed2_cols].sort_values(bed2_cols))
    return BedTool.to_dataframe(bed_1.intersect(bed_2, wa=True, wb=True))
    
def rename_cls_pn(x):
    x = x.replace('0','1').replace('4','3')
    return x

Predictions['sig.comparison'] = Predictions['sig.comparison'].apply(rename_cls_pn)

overlapping_regions = find_overlapping_regions_df(DMRs, Predictions.sort_values('start'), \
                                        [0,1,2,'type','mdiff','c'], ['chr','start','stop','sig.comparison'], )
overlapping_regions['overlapped'] = overlapping_regions.apply(lambda x:(min(x['end'],x['itemRgb'])-max(x['start'],x['thickEnd'])), axis=1)

tlabel = {
    '0,1,2|3,4False': '1|1|1|3|3',
    '0,1,2|3,4LFalse': '1|1|1|3|3',
    '0,1,2|3,4LTrue': '3|3|3|1|1',
    '0,1,2|3,4True': '3|3|3|1|1',
    '0,1|2False': '3|3|1|2|2',
    '0,1|2True': '1|1|3|2|2',
    '0|othersFalse': '1|3|3|3|3',
    '0|othersTrue': '3|1|1|1|1',
    '3|4False': '2|2|2|1|3',
    '3|4True': '2|2|2|3|1'
}

overlapping_regions['tlabel'] = (overlapping_regions['name']+(overlapping_regions['score']>0).astype(str)).map(tlabel)
def ndiff(a,b):
    n = 0
    for ii, i in enumerate(a):
        if i=='x':
            pass
        else:
            if b[ii]!=i:
                n+=1
    return n

overlapping_regions['ndiff'] = overlapping_regions.apply(\
                    lambda x:ndiff(x['tlabel'], x['blockCount']), axis=1)
dtypeaccs = overlapping_regions.sort_values('overlapped', ascending=False).drop_duplicates(subset=['chrom','start','end'],keep='first')\
[['name','strand','ndiff']].groupby(['name','strand']).mean()
dtypeaccs['type'] = [i[0] for i in dtypeaccs.index]
dtypeaccs['c'] = [i[1] for i in dtypeaccs.index]
dtypeaccs['ndiff'] = (5-dtypeaccs['ndiff'])/5

overlapping_regions.index = overlapping_regions['chrom']+'.'+\
                            overlapping_regions['start'].astype(str)+'.'+\
                            overlapping_regions['end'].astype(str)
overlapping_regions['leaf'] = overlapping_regions.index.map(DMRs['leaf'])
overlapping_regions['leaf'] = overlapping_regions['leaf'].fillna('0').apply(lambda x:int(x.split('(')[0])).fillna(0)
L_counts = overlapping_regions.\
groupby(['start','end','leaf','strand']).count()[['name']]
L_counts['lenf'] = L_counts.index.get_level_values(2)
L_counts = pd.crosstab(L_counts['lenf'],L_counts['name'])
L_counts['All'] = L_counts.index.map(DMRs['leaf'].value_counts())
L_counts['lenfint'] = [int(i) for i in L_counts.index]

In [ ]:
(5-overlapping_regions.sort_values('overlapped', ascending=False).drop_duplicates(subset=['chrom','start','end'],keep='first')\
[['name','strand','ndiff']]['ndiff'].mean())/5

In [ ]:
plt.subplots(figsize=(5,5))
sns.heatmap(dtypeaccs.pivot(columns='c',index='type',).loc[
            ['0|others','0,1,2|3,4','0,1|2','3|4','0,1,2|3,4L']
], annot=True, fmt='.3g',
            cmap='Reds', vmin=0,vmax=1)

plt.savefig('./figures/ED2e.pdf',bbox_inches='tight')

In [ ]:
dtypeaccs.pivot(columns='c',index='type',).loc[
            ['0|others','0,1,2|3,4','0,1|2','3|4','0,1,2|3,4L']
]

In [ ]:
plt.subplots(figsize=(5,5))
sns.heatmap(L_counts.sort_values('lenfint').drop(columns=['lenfint','All']),
            annot=True, fmt='.3g',
            cmap='Reds', vmin=0,vmax=20)
plt.savefig('./figures/ED2g.pdf',bbox_inches='tight')

In [ ]:
L_counts.sort_values('lenfint').drop(columns=['lenfint','All'])

In [ ]:
ab='155'
abab = '15_5_10_10'
bg = '1' if ab=='403' else '2'

In [ ]:
DMRs = pd.read_table("./data/"+ab+"-fg-a10/DMRs_"+abab+"_a10.bed", header=None).sort_values([1,2])
DMRs = DMRs.loc[DMRs[6]!='nogrp']
DMRs.index = DMRs[0]+'.'+DMRs[1].astype(str)+'.'+DMRs[2].astype(str)
DMRs['type'] = DMRs[6].apply(lambda x:('0,1,2|3,4L' if x[:4]=='leaf' else x))
DMRs[6] = DMRs[6]+(DMRs[3]>0).map({True:'(P)',False:'(N)'}) +'@'+(DMRs[7]).astype(str)
DMRs['leaf'] = DMRs[6].apply(lambda x:(x.split('|')[1] if x[:4]=='leaf' else None))
DMRs['c'] = DMRs[7]
DMRs['len'] = DMRs[2]-DMRs[1]
DMRs['mdiff'] = DMRs[3]
Predictions = pd.read_table('./DMRs/metilene3.bg'+bg+'-DMRs.tsv')

from pybedtools import BedTool
def find_overlapping_regions_df(bed1_df, bed2_df, bed1_cols, bed2_cols):
    bed_1 = BedTool.from_dataframe(bed1_df[bed1_cols].sort_values(bed1_cols))
    bed_2 = BedTool.from_dataframe(bed2_df[bed2_cols].sort_values(bed2_cols))
    return BedTool.to_dataframe(bed_1.intersect(bed_2, wa=True, wb=True))
    
def rename_cls_pn(x):
    x = x.replace('0','1').replace('4','3')
    return x

Predictions['sig.comparison'] = Predictions['sig.comparison'].apply(rename_cls_pn)

overlapping_regions = find_overlapping_regions_df(DMRs, Predictions.sort_values('start'), \
                                        [0,1,2,'type','mdiff','c'], ['chr','start','stop','sig.comparison'], )
overlapping_regions['overlapped'] = overlapping_regions.apply(lambda x:(min(x['end'],x['itemRgb'])-max(x['start'],x['thickEnd'])), axis=1)

tlabel = {
    '0,1,2|3,4False': '1|1|1|3|3',
    '0,1,2|3,4LFalse': '1|1|1|3|3',
    '0,1,2|3,4LTrue': '3|3|3|1|1',
    '0,1,2|3,4True': '3|3|3|1|1',
    '0,1|2False': '3|3|1|2|2',
    '0,1|2True': '1|1|3|2|2',
    '0|othersFalse': '1|3|3|3|3',
    '0|othersTrue': '3|1|1|1|1',
    '3|4False': '2|2|2|1|3',
    '3|4True': '2|2|2|3|1'
}

overlapping_regions['tlabel'] = (overlapping_regions['name']+(overlapping_regions['score']>0).astype(str)).map(tlabel)
def ndiff(a,b):
    n = 0
    for ii, i in enumerate(a):
        if i=='x':
            pass
        else:
            if b[ii]!=i:
                n+=1
    return n

overlapping_regions['ndiff'] = overlapping_regions.apply(\
                    lambda x:ndiff(x['tlabel'], x['blockCount']), axis=1)
dtypeaccs = overlapping_regions.sort_values('overlapped', ascending=False).drop_duplicates(subset=['chrom','start','end'],keep='first')\
[['name','strand','ndiff']].groupby(['name','strand']).mean()
dtypeaccs['type'] = [i[0] for i in dtypeaccs.index]
dtypeaccs['c'] = [i[1] for i in dtypeaccs.index]
dtypeaccs['ndiff'] = (5-dtypeaccs['ndiff'])/5

overlapping_regions.index = overlapping_regions['chrom']+'.'+\
                            overlapping_regions['start'].astype(str)+'.'+\
                            overlapping_regions['end'].astype(str)
overlapping_regions['leaf'] = overlapping_regions.index.map(DMRs['leaf'])
overlapping_regions['leaf'] = overlapping_regions['leaf'].fillna('0').apply(lambda x:int(x.split('(')[0])).fillna(0)
L_counts = overlapping_regions.\
groupby(['start','end','leaf','strand']).count()[['name']]
L_counts['lenf'] = L_counts.index.get_level_values(2)
L_counts = pd.crosstab(L_counts['lenf'],L_counts['name'])
L_counts['All'] = L_counts.index.map(DMRs['leaf'].value_counts())
L_counts['lenfint'] = [int(i) for i in L_counts.index]

In [ ]:
(5-overlapping_regions.sort_values('overlapped', ascending=False).drop_duplicates(subset=['chrom','start','end'],keep='first')\
[['name','strand','ndiff']]['ndiff'].mean())/5

In [ ]:
plt.subplots(figsize=(5,5))
sns.heatmap(dtypeaccs.pivot(columns='c',index='type',).loc[
            ['0|others','0,1,2|3,4','0,1|2','3|4','0,1,2|3,4L']
], annot=True, fmt='.3g',
            cmap='Reds', vmin=0,vmax=1)

plt.savefig('./figures/ED2e-r.pdf',bbox_inches='tight')

In [ ]:
dtypeaccs.pivot(columns='c',index='type',).loc[
            ['0|others','0,1,2|3,4','0,1|2','3|4','0,1,2|3,4L']
]

In [ ]:
plt.subplots(figsize=(5,5))
sns.heatmap(L_counts.sort_values('lenfint').drop(columns=['lenfint','All']),
            annot=True, fmt='.3g',
            cmap='Reds', vmin=0,vmax=20)
plt.savefig('./figures/ED2g-r.pdf',bbox_inches='tight')

In [ ]:
L_counts.sort_values('lenfint').drop(columns=['lenfint','All'])

In [ ]:
tm = pd.read_table('./DMRs/cpu-memory.tsv')
tm['memory-GB'] = tm['memory-MB']/1024
tm

In [ ]:
plt.subplots(figsize=[3,5])
sns.barplot(data=tm,x='software',y='minutes',hue='#cores',order=['metilene3','wgbstools','smart2'])
plt.yticks([0,30,60])
plt.savefig('./figures/ED2i.pdf', bbox_inches='tight')

In [ ]:
plt.subplots(figsize=[3,5])
sns.barplot(data=tm,x='software',y='memory-MB',hue='#cores',order=['metilene3','wgbstools','smart2'])
plt.yticks([0,512,1024])
plt.savefig('./figures/ED2i-b.pdf', bbox_inches='tight')